In [1]:
# !pip install --upgrade pip
# !python3.7 -m pip install rank_bm25
# !pip install sentence-transformers==2.2.2 transformers==4.28.1 --no-cache-dir
# !pip install faiss-cpu
# !pip install nltk
# !python3.7 -m pip install stanza
# !pip install --upgrade typing_extensions==4.5.0

In [2]:
import pandas as pd
import numpy as np
import re
import pyspark
from pyspark.sql import SparkSession
import shutil
import json
from rank_bm25 import BM25Okapi

import nltk
from nltk import pos_tag
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
stopwords_set = set(stopwords.words('english'))

ModuleNotFoundError: No module named 'pandas'

In [7]:
# pyspark works best with java8 
# set JAVA_HOME enviroment variable to java8 path 
# %env JAVA_HOME = /usr/lib/jvm/java-8-openjdk-amd64

import os
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-8.jdk/Contents/Home"


In [8]:
spark = SparkSession.builder.getOrCreate()
# sc = pyspark.SparkContext()

25/04/22 15:09:58 WARN Utils: Your hostname, MacBook-Pro-2.local resolves to a loopback address: 127.0.0.1; using 172.24.1.190 instead (on interface en0)
25/04/22 15:09:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/22 15:09:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Read Dataset and Preprocessing Data

In [9]:
# Small subset of training dataset - 30% of original data
# Full dataset: s3://nina-rag-project/wiki_movie_plots_deduped.csv

df = spark.read.csv("data/movie_subset.csv", header=True, multiLine=True, escape="\"", quote="\"")
# columns = [
#     "Release Year", "Title", "Origin/Ethnicity", "Director", "Cast", 
#     "Genre", "Wiki Page", "Plot"
# ]
# df = df.toDF(*columns)
df.printSchema()

root
 |-- Release Year: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Origin/Ethnicity: string (nullable = true)
 |-- Director: string (nullable = true)
 |-- Cast: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Wiki Page: string (nullable = true)
 |-- Plot: string (nullable = true)
 |-- plot_length: string (nullable = true)



In [10]:
print(df.count()) 

10466


In [11]:
from pyspark.sql.functions import col, when, count, trim

missing_counts = df.select([
    count(when(col(c).isNull() | (trim(col(c)) == ""), c)).alias(c)
    for c in df.columns
])

missing_counts.show()

+------------+-----+----------------+--------+----+-----+---------+----+-----------+
|Release Year|Title|Origin/Ethnicity|Director|Cast|Genre|Wiki Page|Plot|plot_length|
+------------+-----+----------------+--------+----+-----+---------+----+-----------+
|           0|    0|               0|       0|   0|    5|        0|   0|          0|
+------------+-----+----------------+--------+----+-----+---------+----+-----------+



In [12]:
from pyspark.sql.functions import length

df = df.withColumn("plot_length", length(df["Plot"]))
df.select("plot_length").describe().show()

+-------+------------------+
|summary|       plot_length|
+-------+------------------+
|  count|             10466|
|   mean|2156.6063443531434|
| stddev|1762.6956931513862|
|    min|                15|
|    max|             29442|
+-------+------------------+



In [37]:
CHUNK_SIZE = 800
MIN_CHARS = 200

def split_into_chunks(key, text, chunk_size=CHUNK_SIZE, min_chars=MIN_CHARS):
    text = text.strip()
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
    return [(f"{key}_chunk{i}", chunk) for i, chunk in enumerate(chunks) if len(chunk) >= min_chars]

movie_rdd = movie_rdd.flatMap(lambda x: split_into_chunks(x[0], x[1]))

In [10]:
# # Set this based on combined text columns (value in rdd)
# use_small_dataset = True
# MIN_CHARS = 200 if use_small_dataset else 400
# MAX_CHARS = 2500 if use_small_dataset else 3000

# def truncate_and_filter_by_length(data):
#     key, text = data
#     text = text[:MAX_CHARS]
#     if len(text) >= MIN_CHARS:
#         return (key, text)
#     else:
#         return None

# movie_rdd = movie_rdd.map(truncate_and_filter_by_length).filter(lambda x: x is not None)

In [38]:
print(f'The dataset after cleaning: {movie_rdd.count()} rows left')

The dataset after cleaning: 32811 rows left


In [39]:
movie_rdd.take(1)

[('The Day the Earth Stood Still_1951_chunk0',
  'Movie: The Day the Earth Stood Still Origin: American Genre: science fiction Director: Robert Wise Cast: Michael Rennie, Patricia Neal Plot: When a flying saucer lands in Washington, D.C., the Army quickly surrounds it. A humanoid (Michael Rennie) emerges, announcing that he has come in peace. When he unexpectedly opens a small device, he is shot by a nervous soldier. A tall robot emerges from the saucer and quickly disintegrates the soldiers\' weapons. The alien orders the robot, Gort, to stop. He explains that the now-broken device was a gift for the President which would have enabled him "to study life on the other planets". The alien, Klaatu, is taken to Walter Reed Hospital. After surgery, he uses a salve to quickly heal his wound. Meanwhile, the Army is unable to enter the saucer; Gor')]

In [40]:
shutil.rmtree("data/movie_cleanedtexts.txt", ignore_errors=True)
movie_rdd.map(lambda x: f"{x[0]}\t{x[1]}").saveAsTextFile("data/movie_cleanedtexts.txt")

## Embedding with SentenceTransformer (SBERT)

In [41]:
from sentence_transformers import SentenceTransformer

In [42]:
def embed_partition(partition):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    for key, text in partition:
        yield (key, model.encode(text).tolist())

embedded_rdd = movie_rdd.mapPartitions(embed_partition)

In [43]:
embedded_rdd.take(1)

[('The Day the Earth Stood Still_1951_chunk0',
  [-0.05878842622041702,
   0.04220902547240257,
   0.01666918583214283,
   -0.028218410909175873,
   -0.002098518656566739,
   -0.023032190278172493,
   0.004483320750296116,
   0.027542298659682274,
   0.02744661457836628,
   -0.026652773842215538,
   0.012366505339741707,
   -0.034207846969366074,
   -0.05975799635052681,
   0.056317735463380814,
   0.006520539987832308,
   -0.017331881448626518,
   -0.08752841502428055,
   -0.046606045216321945,
   -0.03432505950331688,
   -0.008593396283686161,
   -0.07489589601755142,
   0.09589502960443497,
   -0.0020130034536123276,
   0.07861468940973282,
   -0.04470586031675339,
   0.0705321803689003,
   0.05742429941892624,
   0.02591555565595627,
   -0.03206824138760567,
   -0.04129621759057045,
   0.011998993344604969,
   0.004212372470647097,
   -0.06785354018211365,
   0.023047981783747673,
   -0.025262601673603058,
   0.03690694272518158,
   0.04967346042394638,
   -0.05788494646549225,
   

In [ ]:
shutil.rmtree("data/movie_embeddings.txt", ignore_errors=True)
embedded_rdd.map(lambda x: f"{x[0]}\t{','.join(map(str, x[1]))}").saveAsTextFile("data/movie_embeddings.txt")

## Collect embedding files into one TSV

In [ ]:
import glob
with open("movie_embeddings.tsv", "w") as outfile:
    for part in sorted(glob.glob("data/movie_embeddings.txt/part-*")):
        with open(part) as infile:
            outfile.write(infile.read())

titles = []
vectors = []

with open("movie_embeddings.tsv", "r") as f:
    for line in f:
        title, vector_str = line.strip().split("\t")
        vector = list(map(float, vector_str.split(",")))
        titles.append(title)
        vectors.append(vector)

embeddings = np.array(vectors).astype("float32")

## Retrieval System (Local on master node, not Spark)

### Dense Retrieval using Faiss

In [ ]:
import faiss

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)
faiss.write_index(index, "movie_rag_index.faiss")

with open("movie_keys.json", "w") as f:
    json.dump(titles, f)

In [ ]:
title_to_text = {}
with open("data/movie_cleanedtexts.txt/part-00000", "r") as f:
    for line in f:
        key, val = line.strip().split("\t", 1)
        title_to_text[key] = val

### Sparse Retrieval using BM25

In [ ]:
plots_filtered = list(title_to_text.values())
tokenized_corpus = [word_tokenize(doc.lower()) for doc in plots_filtered]
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
def sparse_retrieve(query, k=5):
    tokenized_query = word_tokenize(query.lower())
    scores = bm25.get_scores(tokenized_query)
    top_k_indices = np.argsort(scores)[::-1][:k]
    return [plots_filtered[i] for i in top_k_indices], top_k_indices

### Hybrid Retrieval

In [23]:
import stanza
# stanza.download("en")
nlp = stanza.Pipeline("en", processors="tokenize,pos,lemma")

2025-04-22 14:12:17 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2025-04-22 14:12:18 INFO: Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

2025-04-22 14:12:18 INFO: Using device: cpu
2025-04-22 14:12:18 INFO: Loading: tokenize
2025-04-22 14:12:18 INFO: Loading: pos
2025-04-22 14:12:18 INFO: Loading: lemma
2025-04-22 14:12:18 INFO: Done loading processors!


In [ ]:
genre_mapping = {
    "sci-fi": "science fiction",
    "scifi": "science fiction",
    "sci fi": "science fiction",
    "science fiction": "science fiction",
    "romcom": "romantic comedy",
    "bio": "biography",
    "doc": "documentary",
    "anime": "animated",
    "thriller": "thriller",
    "horror": "horror",
    "action": "action",
    "comedy": "comedy",
    "drama": "drama",
    "mystery": "mystery",
    "fantasy": "fantasy",
    "adventure": "adventure",
    "crime": "crime",
    "war": "war",
    "romance": "romance",
    "family": "family",
    "history": "history",
}

def detect_genres(query):
    query_lower = query.lower()
    detected = []
    for variant, canonical in genre_mapping.items():
        if variant in query_lower:
            detected.append(canonical)
    return list(set(detected))

query = "Suggest a science fiction movie, in which aliens come to Earth in 1990s"
print(detect_genres(query))

target_pos = {"NOUN", "PROPN", "ADJ", "NUM"}

def extract_keywords(text):
    doc = nlp(text)

    common_movie_words = {
        "movie", "film", "story", "show", "watch", "scene", "actor", "actress",
        "director", "character", "plot", "cast", "version", "remake", 
        "see", "watching", "episode", "cinema"
    }
    
    all_stopwords = stopwords_set.union(common_movie_words)

    keywords = []
    for sentence in doc.sentences:
        for word in sentence.words:
#             print(f"{word.text}\t{word.lemma}\t{word.upos}")
            if word.upos in target_pos and word.lemma.lower() not in all_stopwords:
                keywords.append(word.lemma.lower())

    return keywords


In [ ]:
def hybrid_retrieve(query, k=5, dense_weight=0.5, sparse_weight=0.5, boost_genre=True, boost_title=True):
    detected_genre = detect_genres(query)
    print("Detected genres:", detected_genre)

    keywords = extract_keywords(query)
    print("Extracted keywords:", keywords)
    processed_query = " ".join(keywords)

    # Encode query with SBERT
    model = SentenceTransformer('all-MiniLM-L6-v2')
    dense_vector = model.encode([processed_query]).astype("float32")
    D, I = index.search(dense_vector, k=10)
    dense_indices = I[0]

    # Sparse retrieval (BM25)
    _, sparse_indices = sparse_retrieve(processed_query, k=10)

    # Combine scores from both retrievals
    combined_scores = {}
    for rank, idx in enumerate(dense_indices):
        combined_scores[idx] = combined_scores.get(idx, 0) + dense_weight * (1 / (1 + rank))
    for rank, idx in enumerate(sparse_indices):
        combined_scores[idx] = combined_scores.get(idx, 0) + sparse_weight * (1 / (1 + rank))

    # Boost scores if genres match
    if boost_genre and detected_genre:
        for i, title in enumerate(titles):
            genre_text = title_to_text.get(title, "").lower()
            if "genre: unknown" in genre_text:
                combined_scores[i] = combined_scores.get(i, 0) - 0.75
            for g in detected_genre:
                if g in genre_text:
                    combined_scores[i] = combined_scores.get(i, 0) + 1.0
                    break

    # Boost if movie title appears in query
    if boost_title:
        query_lower = query.lower()
        for i, title in enumerate(titles):
            if title.lower().split("_")[0] in query_lower:
                combined_scores[i] = combined_scores.get(i, 0) + 2.0

    # Sort final scores
    sorted_indices = sorted(combined_scores, key=combined_scores.get, reverse=True)

    # Filter to keep only movies with matching or unknown genre
    if detected_genre:
        genre_filtered_indices = []
        for i in sorted_indices:
            genre_section = title_to_text.get(titles[i], "").lower()
            if "genre: unknown" in genre_section or any(g in genre_section for g in detected_genre):
                genre_filtered_indices.append(i)
        if genre_filtered_indices:
            sorted_indices = genre_filtered_indices[:k]

    # Final top results
    top_titles = [titles[i] for i in sorted_indices[:k]]
    top_docs = [title_to_text[t] for t in top_titles]

    return top_docs, top_titles


In [ ]:
query = "Suggest a scifi movie, in which aliens come to Earth"
top_docs, top_titles = hybrid_retrieve(query)

print(f"\nTop results for: {query}")
for title, doc in zip(top_titles, top_docs):
    print(f"🎬 {title}")
    print(doc[:300] + "...\n")

## Generation System using T5

In [ ]:
# rm -rf ~/.cache/huggingface/hub/models--google--flan-t5-base

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "google/flan-t5-base"
model = T5ForConditionalGeneration.from_pretrained(model_name)
tokenizer = T5Tokenizer.from_pretrained(model_name)

In [ ]:
# input_text = "translate English to Gernman: How old are you?"
# input_ids = tokenizer(input_text, return_tensors="pt").input_ids

# outputs = model.generate(input_ids)
# print(tokenizer.decode(outputs[0]))

In [ ]:
def generate_answer(query, model=model, tokenizer=tokenizer, max_input_len=512, max_output_len=300):
    context = "\n\n".join([f"Movie: {title}\n{plot}" for title, plot in zip(top_titles[:1], top_docs[:1])])
    prompt = (
        f"You are an expert in movies.\n"
        f"Task: Why is this a good movie recommendation for question: {query}\n\n"
        f"Context: {context}\n\n"
        f"Answer in one full sentence:"
    )
    print(prompt)

    input_ids = tokenizer(prompt, return_tensors="pt", max_length=max_input_len, truncation=False).input_ids
    outputs = model.generate(input_ids, max_new_tokens=max_output_len, num_beams=4, early_stopping=True, no_repeat_ngram_size=2)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

In [ ]:
answer = generate_answer(query)
print(answer)